# XAGUSD H1 Phase 1: Data Foundation

## Objective
- Load raw data and validate quality
- Calculate ATR for multiple periods
- Detect trading sessions and gaps
- Produce clean dataset for backtesting
- Generate baseline statistics

**Status:** Ready to execute  
**Expected Duration:** 5-15 minutes  
**Output:** `data/XAGUSD_H1_CLEAN.csv` + statistics report

## Step 1: Setup & Import

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Imports successful")
print(f"  Pandas: {pd.__version__}")
print(f"  NumPy: {np.__version__}")

## Step 2: Load Raw Data

In [ ]:
# Load raw data
# Adjust path if needed
input_path = '../data/XAGUSD_H1_RAW.csv'

df_raw = pd.read_csv(input_path)
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
df_raw = df_raw.sort_values('timestamp').reset_index(drop=True)

print(f"✓ Loaded {len(df_raw)} candles")
print(f"  Date range: {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")
print(f"  Duration: {(df_raw['timestamp'].max() - df_raw['timestamp'].min()).days} days")
print(f"\nFirst 5 rows:")
print(df_raw.head())
print(f"\nData types:")
print(df_raw.dtypes)

## Step 3: Data Quality Checks

In [ ]:
# OHLC Logic Validation
print("\n" + "="*60)
print("XAGUSD H1 DATA QUALITY REPORT")
print("="*60)

print("\n✓ OHLC LOGIC CHECKS:")

# High >= Low
invalid_hl = (df_raw['high'] < df_raw['low']).sum()
print(f"  High < Low........................... {invalid_hl} rows")

# Close in range
invalid_c = ((df_raw['close'] > df_raw['high']) | (df_raw['close'] < df_raw['low'])).sum()
print(f"  Close outside H/L range............. {invalid_c} rows")

# Open in range
invalid_o = ((df_raw['open'] > df_raw['high']) | (df_raw['open'] < df_raw['low'])).sum()
print(f"  Open outside H/L range.............. {invalid_o} rows")

# Zero range
zero_range = ((df_raw['open'] == df_raw['high']) & 
              (df_raw['high'] == df_raw['low']) & 
              (df_raw['low'] == df_raw['close'])).sum()
print(f"  Zero-range candles.................. {zero_range} rows")

# Volume
zero_vol = (df_raw['volume'] == 0).sum()
print(f"  Zero-volume candles................. {zero_vol} rows (normal for metals)")

print(f"\n✓ Quality verdict: {'PASS' if all([invalid_hl==0, invalid_c==0, invalid_o==0, zero_range==0]) else 'ISSUES FOUND'}")

## Step 4: Detect Gaps & Sessions

In [ ]:
# Detect gaps in timestamp sequence
print("\n⏱ TIMESTAMP & SESSION CHECKS:")
print(f"  Total candles......................... {len(df_raw)}")

gaps = []
for i in range(1, len(df_raw)):
    delta = df_raw['timestamp'].iloc[i] - df_raw['timestamp'].iloc[i-1]
    if delta > pd.Timedelta(hours=1):
        gaps.append({
            'from': df_raw['timestamp'].iloc[i-1],
            'to': df_raw['timestamp'].iloc[i],
            'hours': int(delta.total_seconds() / 3600)
        })

print(f"  Gap periods detected (>1h)........... {len(gaps)}")
if len(gaps) > 0:
    print(f"\n  First 5 gaps:")
    for gap in gaps[:5]:
        print(f"    {gap['from']} → {gap['to']} ({gap['hours']}h)")

# Session classification
utc_hour = df_raw['timestamp'].dt.hour
day_of_week = df_raw['timestamp'].dt.dayofweek

sessions = []
for i in range(len(df_raw)):
    hour = utc_hour.iloc[i]
    dow = day_of_week.iloc[i]
    
    if dow >= 5:
        sessions.append('WEEKEND')
    elif 7 <= hour < 16:
        sessions.append('LONDON')
    elif 12 <= hour < 21:
        sessions.append('US')
    else:
        sessions.append('ASIA')

df_raw['session'] = sessions

print("\n  Session distribution:")
for session in df_raw['session'].unique():
    count = (df_raw['session'] == session).sum()
    pct = 100 * count / len(df_raw)
    print(f"    {session:10s}........................ {count:6d} ({pct:5.1f}%)")

## Step 5: Calculate ATR

In [ ]:
# Calculate True Range
high_low = df_raw['high'] - df_raw['low']
high_pc = abs(df_raw['high'] - df_raw['close'].shift(1))
low_pc = abs(df_raw['low'] - df_raw['close'].shift(1))

df_raw['true_range'] = high_low.combine(high_pc, max).combine(low_pc, max)
df_raw.loc[0, 'true_range'] = high_low.iloc[0]

print("✓ True Range calculated")

# Calculate ATR for multiple periods
for period in [7, 14, 21, 28]:
    df_raw[f'ATR_{period}'] = df_raw['true_range'].rolling(window=period).mean()

print("✓ ATR(7, 14, 21, 28) calculated")

# Calculate candle metrics
df_raw['candle_range'] = df_raw['high'] - df_raw['low']
df_raw['candle_body'] = abs(df_raw['close'] - df_raw['open'])
df_raw['candle_wick_upper'] = df_raw['high'] - df_raw[['open', 'close']].max(axis=1)
df_raw['candle_wick_lower'] = df_raw[['open', 'close']].min(axis=1) - df_raw['low']

print("✓ Candle structure metrics calculated")

print("\nEnriched columns:")
print(df_raw.columns.tolist())

## Step 6: Baseline Statistics

In [ ]:
print("\n" + "="*60)
print("BASELINE STATISTICS")
print("="*60)

stats = {
    'Metric': [],
    'Value': []
}

# Price range
stats['Metric'].append('Price Range - Min')
stats['Value'].append(f"${df_raw['low'].min():.2f}")

stats['Metric'].append('Price Range - Max')
stats['Value'].append(f"${df_raw['high'].max():.2f}")

stats['Metric'].append('Price Range - Mean')
stats['Value'].append(f"${df_raw['close'].mean():.2f}")

# Candle structure
stats['Metric'].append('Candle Range (H-L) - Mean')
stats['Value'].append(f"${df_raw['candle_range'].mean():.4f}")

stats['Metric'].append('Candle Range (H-L) - Median')
stats['Value'].append(f"${df_raw['candle_range'].median():.4f}")

stats['Metric'].append('Candle Range (H-L) - Std Dev')
stats['Value'].append(f"${df_raw['candle_range'].std():.4f}")

stats['Metric'].append('Candle Body - Mean')
stats['Value'].append(f"${df_raw['candle_body'].mean():.4f}")

# ATR statistics
for period in [7, 14, 21, 28]:
    atr_col = f'ATR_{period}'
    atr_clean = df_raw[atr_col].dropna()
    
    stats['Metric'].append(f'ATR({period}) - Mean')
    stats['Value'].append(f"${atr_clean.mean():.4f}")
    
    stats['Metric'].append(f'ATR({period}) - Median')
    stats['Value'].append(f"${atr_clean.median():.4f}")
    
    stats['Metric'].append(f'ATR({period}) - Std Dev')
    stats['Value'].append(f"${atr_clean.std():.4f}")
    
    stats['Metric'].append(f'ATR({period}) - Min')
    stats['Value'].append(f"${atr_clean.min():.4f}")
    
    stats['Metric'].append(f'ATR({period}) - Max')
    stats['Value'].append(f"${atr_clean.max():.4f}")

df_stats = pd.DataFrame(stats)
print("\n" + df_stats.to_string(index=False))

# Save statistics
stats_path = '../reports/00_baseline_statistics.csv'
df_stats.to_csv(stats_path, index=False)
print(f"\n✓ Statistics saved to {stats_path}")

## Step 7: Volatility Visualization

In [ ]:
# Create 3-panel volatility profile chart
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# Panel 1: Price over time
ax1 = axes[0]
ax1.plot(df_raw['timestamp'], df_raw['close'], label='Close', linewidth=1, color='#1f77b4')
ax1.fill_between(df_raw['timestamp'], df_raw['low'], df_raw['high'], alpha=0.2, color='#1f77b4')
ax1.set_ylabel('Price (USD/oz)', fontsize=11, fontweight='bold')
ax1.set_title('XAGUSD H1 - Price Action', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Panel 2: ATR(14) over time
ax2 = axes[1]
atr14 = df_raw['ATR_14'].dropna()
ax2.plot(df_raw['timestamp'][df_raw['ATR_14'].notna()], atr14, label='ATR(14)', linewidth=1.5, color='#ff7f0e')
ax2.fill_between(df_raw['timestamp'][df_raw['ATR_14'].notna()], 0, atr14, alpha=0.3, color='#ff7f0e')
ax2.axhline(atr14.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${atr14.mean():.4f}')
ax2.axhline(atr14.median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${atr14.median():.4f}')
ax2.set_ylabel('ATR(14) [USD/oz]', fontsize=11, fontweight='bold')
ax2.set_title('Volatility Profile - ATR(14)', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Panel 3: ATR Distribution (histogram)
ax3 = axes[2]
ax3.hist(atr14, bins=50, color='#2ca02c', alpha=0.7, edgecolor='black')
ax3.axvline(atr14.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${atr14.mean():.4f}')
ax3.axvline(atr14.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: ${atr14.median():.4f}')
ax3.axvline(atr14.quantile(0.75), color='purple', linestyle=':', linewidth=2, label=f'75th %ile: ${atr14.quantile(0.75):.4f}')
ax3.axvline(atr14.quantile(0.25), color='purple', linestyle=':', linewidth=2, label=f'25th %ile: ${atr14.quantile(0.25):.4f}')
ax3.set_xlabel('ATR(14) [USD/oz]', fontsize=11, fontweight='bold')
ax3.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax3.set_title('ATR(14) Distribution', fontsize=13, fontweight='bold')
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig('../reports/01_volatility_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Volatility chart saved to reports/01_volatility_profile.png")

## Step 8: Prepare Clean Data for Backtesting

In [ ]:
# Remove warmup period (where ATR is still converging)
warmup_period = 30
df_clean = df_raw[df_raw.index >= warmup_period].reset_index(drop=True)

print(f"✓ Warmup period: {warmup_period} candles removed")
print(f"  Raw data candles..................... {len(df_raw)}")
print(f"  Backtesting candles.................. {len(df_clean)}")

# Select columns for backtesting
backtest_columns = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume',
    'true_range', 'candle_range', 'candle_body',
    'candle_wick_upper', 'candle_wick_lower',
    'ATR_7', 'ATR_14', 'ATR_21', 'ATR_28', 'session'
]

df_clean = df_clean[backtest_columns]

print(f"\n✓ Selected backtesting columns ({len(backtest_columns)}):")
for i, col in enumerate(backtest_columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nClean data ready:")
print(f"  Rows: {len(df_clean)}")
print(f"  Columns: {len(df_clean.columns)}")
print(f"  Memory: {df_clean.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"\nFirst 5 rows of clean data:")
print(df_clean.head())

## Step 9: Save Clean Data

In [ ]:
output_path = '../data/XAGUSD_H1_CLEAN.csv'
df_clean.to_csv(output_path, index=False)

print(f"✓ Clean data saved to {output_path}")
print(f"\nPhase 1 Complete!")
print(f"\nNext Steps:")
print(f"  1. Review baseline statistics: reports/00_baseline_statistics.csv")
print(f"  2. Review volatility chart: reports/01_volatility_profile.png")
print(f"  3. Proceed to Phase 2: Pattern Detection")
print(f"\n" + "="*60)
print(f"Ready for backtesting: {output_path}")
print(f"="*60)